# Session 1: Retrieval-Augmented Generation (Instructor Notebook)

Welcome to Day 3! In this session, students build production-grade RAG systems for **McKinsey's consulting knowledge management**. Moving beyond simple keyword matching, they learn embedding-based retrieval, vector stores, advanced chunking, query transformation, and evaluation -- the techniques that power McKinsey's internal knowledge bases, engagement playbooks, and industry insight repositories.

> **Instructor Note:** This notebook contains all 5 demos (identical to student) plus fully solved versions of all 4 tasks with approach explanations. All examples use McKinsey consulting scenarios: strategy reports, post-merger integration briefs, industry whitepapers, and engagement case studies.

## Learning Objectives

By the end of this session, students will be able to:

1. **Create and compare** text embeddings using OpenAI's embedding models on consulting knowledge artifacts
2. **Build and query** a vector store (ChromaDB) to power a McKinsey knowledge base
3. **Apply advanced chunking** strategies to consulting documents (strategy reports, engagement briefs, whitepapers)
4. **Transform queries** using HyDE and multi-query techniques for comprehensive industry analysis
5. **Build an end-to-end RAG pipeline** with source citations over McKinsey engagement materials
6. **Evaluate RAG quality** with retrieval and generation metrics for consulting use cases

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Embedding Models -- Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In a consulting context, embeddings allow us to find relevant strategy insights, industry analyses, and engagement playbooks even when the exact terminology differs -- e.g., a query about "post-merger synergies" should retrieve documents discussing "integration value capture."

In [ ]:
# Demo 1 - Embedding Models (McKinsey Consulting Knowledge)

client = openai.OpenAI()

# Step 1: Create embeddings for McKinsey consulting knowledge snippets
texts = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and cultural alignment between merging entities.",
    "Omnichannel retail strategy must prioritize seamless customer experience across physical and digital touchpoints.",
    "Supply chain resilience requires diversification of sourcing and near-shoring of critical components.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")

# Step 2: Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nSimilarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

# Step 3: Semantic search — consulting query
query = "What are best practices for post-merger integration?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

print(f"\nQuery: {query}")
print("Ranked results:")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

### Demo 2: Vector Stores -- Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

In this demo, we build a **McKinsey knowledge base** -- indexing consulting insights across strategy, operations, M&A, digital transformation, and organizational design so consultants can quickly surface relevant prior work.

In [ ]:
# Demo 2 - Vector Stores with ChromaDB (McKinsey Knowledge Base)

# Step 1: Create a ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="mckinsey_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

# Step 2: Add McKinsey consulting knowledge documents with metadata
documents = [
    "Digital transformation in financial services requires a phased approach: digitize core processes, then build new digital offerings, and finally reimagine the business model.",
    "Post-merger integration should follow a 100-day plan covering Day-1 readiness, synergy capture, cultural integration, and operating model design.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across all channels to deliver a seamless experience.",
    "Private equity value creation levers include revenue growth acceleration, operational efficiency, strategic M&A, and balance sheet optimization.",
    "Organizational restructuring requires a clear target operating model, role clarity, and a change management program to drive adoption.",
    "ESG strategy should be embedded into core business operations, not treated as a standalone compliance exercise, to create long-term shareholder value.",
    "Healthcare cost transformation requires addressing clinical variation, supply chain optimization, and administrative simplification simultaneously.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, cultural, and operational integration risks."
]

# Use OpenAI embeddings via the API
client = openai.OpenAI()
emb_response = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=documents)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[
        {"source": "digital_transformation_playbook", "practice": "Digital"},
        {"source": "ma_integration_guide", "practice": "M&A"},
        {"source": "retail_strategy_whitepaper", "practice": "Retail"},
        {"source": "pe_value_creation_framework", "practice": "Private Equity"},
        {"source": "org_design_handbook", "practice": "Organization"},
        {"source": "esg_strategy_report", "practice": "Sustainability"},
        {"source": "healthcare_cost_study", "practice": "Healthcare"},
        {"source": "cross_border_ma_guide", "practice": "M&A"},
    ]
)
print(f"Indexed {collection.count()} documents in McKinsey knowledge base")

# Step 3: Query the collection — a typical consultant research question
query = "How should a retailer approach omnichannel transformation?"
query_emb = client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"\nQuery: {query}")
print("Top 3 results:")
for i, (doc, distance, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta['source']} | {meta['practice']}] (dist={distance:.4f}) {doc}")

### Demo 3: Advanced Chunking Strategies

How you split documents directly affects retrieval quality. Different strategies work better for different content types: recursive splitting for general text, section-aware splitting for structured consulting reports, and sentence-based for executive summaries.

McKinsey documents typically have a structured format: **Executive Summary, Key Findings, Detailed Analysis, Recommendations, and Appendices**. Choosing the right chunking strategy preserves the logical structure of these sections.

In [ ]:
# Demo 3 - Advanced Chunking Strategies (McKinsey Consulting Report)

sample_doc = """# Post-Merger Integration: A Best Practice Guide

Post-merger integration (PMI) is the critical phase that determines whether an M&A transaction delivers its intended value. McKinsey research shows that 70% of mergers fail to achieve projected synergies, most often due to integration execution failures.

## Executive Summary

Successful post-merger integration requires a structured 100-day plan, early synergy identification, cultural alignment, and disciplined execution. Our analysis of 200+ transactions reveals that Day-1 readiness is the single strongest predictor of integration success.

## Key Findings

Three factors distinguish successful integrations from unsuccessful ones:

1. Leadership alignment: Joint leadership teams must be appointed within the first two weeks, with clear decision rights and accountability.
2. Synergy capture: Revenue and cost synergies must be quantified with bottom-up rigor and tracked through a dedicated Integration Management Office (IMO).
3. Cultural integration: Cultural differences must be proactively addressed through joint team workshops, shared values articulation, and visible leadership behavior.

## Recommendations

We recommend a phased approach to integration. Phase 1 (Days 1-30) focuses on stabilization and quick wins. Phase 2 (Days 31-100) addresses operating model design and synergy capture. Phase 3 (Months 4-12) drives full integration and transformation.

## Appendix

Detailed case studies from recent financial services and healthcare M&A transactions are provided in the supplementary materials, including synergy tracking templates and cultural assessment frameworks."""

# Strategy 1: Fixed-size recursive splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "200")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")), separators=["\n\n", "\n", ". ", " "]
)
recursive_chunks = recursive_splitter.split_text(sample_doc)

print(f"Recursive splitting: {len(recursive_chunks)} chunks")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 2: Section-aware splitting (respects consulting report structure)
section_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")),
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
section_chunks = section_splitter.split_text(sample_doc)

print(f"\nSection-aware splitting: {len(section_chunks)} chunks")
for i, chunk in enumerate(section_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 3: Sentence-level splitting (good for executive summaries)
sentence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=int(os.getenv("CHUNK_SIZE", "150")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "0")),
    separators=[". ", "\n", " "]
)
sentence_chunks = sentence_splitter.split_text(sample_doc)

print(f"\nSentence-level splitting: {len(sentence_chunks)} chunks")
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

print(f"\nComparison: Recursive={len(recursive_chunks)}, Section-aware={len(section_chunks)}, Sentence={len(sentence_chunks)}")

### Demo 4: Query Transformation Techniques

Consultants often write queries that are too vague or too specific for direct retrieval against the knowledge base. Query transformation improves retrieval by rewriting, expanding, or hypothetically answering the query before searching.

For example, a consultant asking "How do we help a bank go digital?" needs the system to also search for "core banking modernization," "digital customer experience," and "fintech partnership strategy."

In [ ]:
# Demo 4 - Query Transformation (McKinsey Consulting Research)

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

# Technique 1: Multi-query expansion for comprehensive industry analysis
def multi_query_expand(question, n=3):
    """Generate multiple alternative queries for better retrieval coverage."""
    response = llm.invoke([
        SystemMessage(content=f"You are a McKinsey consultant researching a topic. Generate {n} alternative versions of this research question. Each should approach it from a different angle (e.g., industry perspective, functional perspective, geographic perspective). Return as a JSON list of strings."),
        HumanMessage(content=question)
    ])
    try:
        queries = json.loads(response.content)
    except:
        queries = [question]
    return [question] + queries

# Technique 2: HyDE (Hypothetical Document Embeddings)
def hyde_transform(question):
    """Generate a hypothetical McKinsey insight, then use it for retrieval."""
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey senior partner. Write a short paragraph that would be a perfect answer to this consulting question. Write it as if it's from a McKinsey whitepaper -- authoritative, structured, and insight-driven."),
        HumanMessage(content=question)
    ])
    return response.content

# Test both techniques — typical consulting research question
question = "What are the key risks in cross-border M&A transactions?"

print(f"Original question: {question}")
print("\n--- Multi-Query Expansion ---")
expanded = multi_query_expand(question)
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

print("\n--- HyDE Transform ---")
hypothetical = hyde_transform(question)
print(f"  Hypothetical McKinsey insight: {hypothetical[:200]}...")
print("\n  (This hypothetical answer would be embedded and used for retrieval")
print("   instead of the original question, often finding more relevant chunks.)")

### Demo 5: End-to-End RAG Pipeline with Source Citations

This demo brings everything together into a complete RAG pipeline that retrieves relevant McKinsey knowledge artifacts, generates a consultant-ready answer, and includes source citations -- just as a consultant would reference prior engagement materials and published research in their deliverables.

In [ ]:
# Demo 5 - End-to-End RAG Pipeline with Citations (McKinsey Knowledge Base)

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
embed_client = openai.OpenAI()

# Step 1: Build McKinsey knowledge base
kb_docs = [
    {"text": "Post-merger integration success depends on Day-1 readiness. Our research across 200+ deals shows that companies with a structured 100-day plan capture 15-25% more synergies.", "source": "ma_integration_playbook.pdf", "page": 1},
    {"text": "Digital transformation in banking requires three horizons: digitize existing processes (H1), launch digital-native products (H2), and reimagine the business model around platforms (H3).", "source": "digital_banking_whitepaper.pdf", "page": 4},
    {"text": "Supply chain resilience starts with visibility. Tier-2 and Tier-3 supplier mapping, combined with scenario-based stress testing, reduces disruption impact by 30-40%.", "source": "supply_chain_resilience_report.pdf", "page": 7},
    {"text": "ESG-driven value creation requires embedding sustainability metrics into strategic planning, capital allocation, and executive compensation to move beyond compliance.", "source": "esg_strategy_guide.pdf", "page": 2},
    {"text": "Organizational health is the strongest predictor of long-term performance. Companies in the top quartile of OHI scores deliver 3x total returns to shareholders.", "source": "ohi_benchmark_study.pdf", "page": 12},
    {"text": "Revenue synergies in mergers are realized through cross-selling, pricing optimization, and geographic expansion, but require dedicated commercial integration teams to execute.", "source": "ma_integration_playbook.pdf", "page": 15}
]

# Step 2: Index in ChromaDB
chroma = chromadb.Client()
coll = chroma.create_collection(name="engagement_playbooks", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in kb_docs]
embs = [e.embedding for e in embed_client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=texts).data]
coll.add(
    documents=texts, embeddings=embs,
    ids=[f"doc_{i}" for i in range(len(kb_docs))],
    metadatas=[{"source": d["source"], "page": d["page"]} for d in kb_docs]
)

# Step 3: RAG function
def rag_query(question, k=3):
    # Retrieve
    q_emb = embed_client.embeddings.create(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[question]).data[0].embedding
    results = coll.query(query_embeddings=[q_emb], n_results=k)
    
    # Build context with source tags
    context_parts = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_parts.append(f"[Source: {meta['source']}, p.{meta['page']}] {doc}")
    context = "\n\n".join(context_parts)
    
    # Generate
    response = llm.invoke([
        SystemMessage(content="You are a McKinsey knowledge assistant. Answer the consultant's question based on the retrieved knowledge base context. Cite sources in [brackets]. If the context does not fully address the question, say so."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
    ])
    
    return {"answer": response.content, "sources": results['metadatas'][0], "context": context}

# Step 4: Test with consulting queries
for q in ["What are best practices for post-merger integration?", "How should a bank approach digital transformation?", "What drives organizational health?"]:
    result = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {[s['source'] for s in result['sources']]}\n")

---
## Tasks (Solved)

### Task 1: Build an Embedding-Based Consulting Knowledge Search Engine

Create a search engine for McKinsey's consulting knowledge base. It should take a corpus of consulting insights (strategy reports, engagement briefs, industry analyses), embed them, and return the most relevant documents for a consultant's research query -- ranked by cosine similarity.

> **Approach:** We encapsulate embedding creation and search in a class. The `__init__` method embeds all documents once, and `search` embeds the query and computes cosine similarity against all stored embeddings, returning the top-k results sorted by score.

In [ ]:
# Task 1 — SOLUTION: Embedding-Based Consulting Knowledge Search Engine

class SearchEngine:
    def __init__(self, documents):
        self.documents = documents
        self.client = openai.OpenAI()
        # Embed all consulting knowledge documents at init time
        response = self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
            input=documents
        )
        self.embeddings = [item.embedding for item in response.data]
        print(f"Indexed {len(self.documents)} consulting knowledge documents ({len(self.embeddings[0])} dimensions)")

    def _cosine_similarity(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def search(self, query, k=3):
        # Embed the query
        query_emb = self.client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]
        ).data[0].embedding
        
        # Compute similarity against all documents
        scored = [
            (self._cosine_similarity(query_emb, doc_emb), doc)
            for doc_emb, doc in zip(self.embeddings, self.documents)
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:k]


# Test with McKinsey consulting knowledge corpus
corpus = [
    "Digital transformation in financial services requires a phased approach starting with core banking modernization.",
    "Post-merger integration success depends on Day-1 readiness and a structured 100-day plan for synergy capture.",
    "Omnichannel retail strategy must unify inventory, pricing, and customer data across physical and digital channels.",
    "Private equity value creation levers include revenue growth, operational efficiency, and strategic bolt-on acquisitions.",
    "Supply chain resilience requires diversification of sourcing, near-shoring critical components, and end-to-end visibility.",
    "Organizational health, as measured by the OHI, is the strongest predictor of sustained long-term performance.",
    "ESG strategy must be embedded into core business operations and capital allocation to create shareholder value.",
    "Cross-border M&A transactions require careful due diligence on regulatory, tax, and cultural integration risks."
]

engine = SearchEngine(corpus)

for query in ["What are best practices for post-merger integration?", "How should a retailer approach omnichannel transformation?", "What are the key risks in cross-border M&A?"]:
    print(f"\nQuery: {query}")
    results = engine.search(query, k=3)
    for score, doc in results:
        print(f"  {score:.4f} | {doc}")

### Task 2: Implement a Multi-Strategy Chunking Pipeline for Consulting Documents

Build a chunking pipeline that applies different strategies based on consulting document type (strategy report with markdown headers, code/analytics scripts, plain-text engagement notes) and evaluates chunk quality.

> **Approach:** We detect the document type by examining content patterns (markdown headers for strategy reports, Python keywords for analytics code, etc.), then apply an appropriate `RecursiveCharacterTextSplitter` with type-specific separators. Each chunk is returned with metadata about its type and character count.

In [ ]:
# Task 2 — SOLUTION: Multi-Strategy Chunking Pipeline for Consulting Documents

class SmartChunker:
    def __init__(self, chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50"))):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def _detect_type(self, text):
        """Detect document type based on content patterns."""
        lines = text.strip().split("\n")
        header_count = sum(1 for l in lines if l.startswith("#"))
        code_keywords = ["def ", "class ", "import ", "return ", "if __name__"]
        code_count = sum(1 for kw in code_keywords if kw in text)
        
        if header_count >= 2:
            return "strategy_report"
        elif code_count >= 2:
            return "analytics_code"
        else:
            return "engagement_notes"

    def _get_splitter(self, doc_type):
        """Return appropriate splitter for the document type."""
        if doc_type == "strategy_report":
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\n## ", "\n# ", "\n### ", "\n\n", "\n", ". ", " "]
            )
        elif doc_type == "analytics_code":
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\nclass ", "\ndef ", "\n\n", "\n", " "]
            )
        else:
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\n\n", "\n", ". ", " "]
            )

    def chunk(self, text):
        """Chunk text using the appropriate strategy."""
        doc_type = self._detect_type(text)
        splitter = self._get_splitter(doc_type)
        raw_chunks = splitter.split_text(text)
        
        return [
            {"text": chunk, "type": doc_type, "chars": len(chunk), "index": i}
            for i, chunk in enumerate(raw_chunks)
        ]


# Test with different McKinsey consulting document types
chunker = SmartChunker(chunk_size=int(os.getenv("CHUNK_SIZE", "200")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "30")))

# 1. Strategy report (markdown with sections)
strategy_report = """# Digital Transformation in Financial Services

Digital transformation is reshaping the financial services industry at an unprecedented pace.

## Executive Summary

Banks that embrace digital-first strategies achieve 20-30% higher customer satisfaction and 15% lower cost-to-income ratios.

## Key Findings

Legacy core banking systems remain the primary barrier to digital transformation. Cloud migration must precede product innovation.

## Recommendations

Adopt a three-horizon approach: digitize existing processes, launch digital products, and reimagine the business model."""

# 2. Analytics code
analytics_code = """import pandas as pd
from sklearn.cluster import KMeans

def segment_clients(df: pd.DataFrame, n_segments: int = 4) -> pd.DataFrame:
    features = df[['revenue', 'engagement_score', 'deal_size']]
    model = KMeans(n_clusters=n_segments)
    df['segment'] = model.fit_predict(features)
    return df

class EngagementAnalyzer:
    def __init__(self, data_path):
        self.data = pd.read_csv(data_path)
        self.segments = None
    
    def run_segmentation(self):
        self.segments = segment_clients(self.data)
        return self.segments"""

# 3. Plain-text engagement notes
engagement_notes = """The client is a mid-size European pharmaceutical company facing margin pressure from generic competition. Current EBITDA margins have declined from 22% to 16% over the past three years. The CEO has asked McKinsey to identify 200-300 basis points of margin improvement through operational efficiency and portfolio optimization. Initial diagnostic suggests significant opportunity in procurement consolidation and manufacturing network rationalization."""

for label, doc in [("Strategy Report", strategy_report), ("Analytics Code", analytics_code), ("Engagement Notes", engagement_notes)]:
    chunks = chunker.chunk(doc)
    print(f"\n{label} ({chunks[0]['type']}): {len(chunks)} chunks")
    for c in chunks:
        print(f"  [{c['index']}] ({c['chars']} chars): {c['text'][:70]}...")

### Task 3: Create a Query Expansion and Reranking System for Consulting Research

Build a retrieval system that takes a consultant's research question, expands it into multiple variants (covering different industry angles, functional perspectives, and geographic contexts), retrieves candidates for each, deduplicates, and reranks using the LLM.

> **Approach:** We use the multi-query expansion from Demo 4 to generate alternative formulations, retrieve from ChromaDB for each, deduplicate by document text, then use the LLM as a judge to score relevance (0-10) for each candidate. Finally, we sort by score and return the top-k.

In [ ]:
# Task 3 — SOLUTION: Query Expansion and Reranking for Consulting Research

class AdvancedRetriever:
    def __init__(self, collection):
        self.collection = collection
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.embed_client = openai.OpenAI()

    def _expand_query(self, question, n=3):
        """Generate alternative query formulations for comprehensive research coverage."""
        response = self.llm.invoke([
            SystemMessage(content=f"You are a McKinsey consultant expanding a research query. Generate {n} alternative versions of this question. Each should approach it from a different angle (e.g., industry-specific, functional, geographic, or historical perspective). Return as a JSON list of strings."),
            HumanMessage(content=question)
        ])
        try:
            queries = json.loads(response.content)
        except:
            queries = [question]
        return [question] + queries

    def _retrieve_candidates(self, queries, per_query_k=3):
        """Retrieve and deduplicate candidates from all query variants."""
        seen = set()
        candidates = []
        for query in queries:
            q_emb = self.embed_client.embeddings.create(
                model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[query]
            ).data[0].embedding
            results = self.collection.query(query_embeddings=[q_emb], n_results=per_query_k)
            for doc in results['documents'][0]:
                if doc not in seen:
                    seen.add(doc)
                    candidates.append(doc)
        return candidates

    def _rerank(self, question, candidates):
        """Use LLM to score relevance of each candidate to the consulting question."""
        scored = []
        for doc in candidates:
            response = self.llm.invoke([
                SystemMessage(content="You are a McKinsey knowledge manager. Rate the relevance of this document to the consultant's research question on a scale of 0-10. Return ONLY the number."),
                HumanMessage(content=f"Research Question: {question}\n\nDocument: {doc}")
            ])
            try:
                score = int(response.content.strip())
            except:
                score = 0
            scored.append((doc, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def retrieve(self, query, k=3):
        """Full pipeline: expand -> retrieve -> deduplicate -> rerank."""
        print(f"Research Query: {query}")
        
        # Step 1: Expand
        queries = self._expand_query(query)
        print(f"Expanded to {len(queries)} query variants")
        
        # Step 2: Retrieve + deduplicate
        candidates = self._retrieve_candidates(queries)
        print(f"Retrieved {len(candidates)} unique knowledge artifacts")
        
        # Step 3: Rerank
        ranked = self._rerank(query, candidates)
        return ranked[:k]


# Test — reuse the McKinsey knowledge base collection from Demo 2
retriever = AdvancedRetriever(collection)
results = retriever.retrieve("How should we advise a PE portfolio company on operational improvement?")

print("\nReranked results:")
for doc, score in results:
    print(f"  {score}/10 | {doc[:80]}")

### Task 4: Build a Production RAG Pipeline with Evaluation Metrics for Consulting QA

Build a complete RAG system over McKinsey's consulting knowledge base that includes automated evaluation -- measuring retrieval relevance (are the retrieved engagement materials related to the question?), answer faithfulness (is the answer supported by the retrieved sources?), and completeness (does the answer fully address the consultant's question?).

> **Approach:** We build a full RAG pipeline class that wraps indexing, retrieval, and generation. After generating an answer, we use LLM-as-judge to evaluate three metrics on a 1-5 scale: retrieval relevance, faithfulness, and completeness. This pattern is standard for RAG evaluation in production consulting knowledge systems.

In [ ]:
# Task 4 — SOLUTION: Production RAG Pipeline with Evaluation (McKinsey Knowledge QA)

class EvaluatedRAG:
    def __init__(self, documents):
        self.llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))
        self.embed_client = openai.OpenAI()
        
        # Index consulting knowledge documents in ChromaDB
        self.chroma = chromadb.Client()
        self.coll = self.chroma.create_collection(
            name="industry_insights", metadata={"hnsw:space": "cosine"}
        )
        embs = [e.embedding for e in self.embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=documents
        ).data]
        self.coll.add(
            documents=documents, embeddings=embs,
            ids=[f"doc_{i}" for i in range(len(documents))]
        )
        print(f"Indexed {len(documents)} consulting knowledge documents")

    def query(self, question, k=3):
        """Retrieve, generate, and evaluate."""
        # Retrieve
        q_emb = self.embed_client.embeddings.create(
            model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"), input=[question]
        ).data[0].embedding
        results = self.coll.query(query_embeddings=[q_emb], n_results=k)
        context = "\n\n".join(results['documents'][0])
        
        # Generate
        response = self.llm.invoke([
            SystemMessage(content="You are a McKinsey knowledge assistant. Answer the consultant's question based on the retrieved context. Be structured and insight-driven. If the context does not fully address the question, say so."),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
        ])
        answer = response.content
        
        # Evaluate
        metrics = self.evaluate(question, answer, context)
        
        return {"answer": answer, "context": context, "metrics": metrics}

    def evaluate(self, question, answer, context):
        """Evaluate RAG quality using LLM-as-judge."""
        metrics = {}
        
        eval_prompts = {
            "relevance": f"Rate 1-5: How relevant is this context to the consulting question?\n\nQuestion: {question}\n\nContext: {context}",
            "faithfulness": f"Rate 1-5: Is the answer fully supported by the context (no hallucinations or unsupported claims)?\n\nContext: {context}\n\nAnswer: {answer}",
            "completeness": f"Rate 1-5: Does the answer fully address the consultant's question with actionable insights?\n\nQuestion: {question}\n\nAnswer: {answer}"
        }
        
        for metric_name, prompt in eval_prompts.items():
            response = self.llm.invoke([
                SystemMessage(content="You are a strict evaluator of consulting knowledge quality. Return ONLY a number 1-5."),
                HumanMessage(content=prompt)
            ])
            try:
                metrics[metric_name] = int(response.content.strip())
            except:
                metrics[metric_name] = 3  # default
        
        metrics["overall"] = round(sum(metrics.values()) / len(metrics), 2)
        return metrics


# Test with McKinsey consulting knowledge
docs = [
    "Post-merger integration success depends on Day-1 readiness. Companies with a structured 100-day plan capture 15-25% more synergies than those without.",
    "Digital transformation in banking requires a three-horizon approach: digitize core processes, build digital products, and reimagine the business model.",
    "Supply chain resilience starts with end-to-end visibility. Tier-2 and Tier-3 supplier mapping reduces disruption impact by 30-40%.",
    "Organizational health, as measured by the OHI, is the strongest predictor of long-term performance. Top-quartile companies deliver 3x total shareholder returns.",
    "ESG strategy must be integrated into core business operations and capital allocation decisions to create sustainable long-term shareholder value."
]

rag = EvaluatedRAG(docs)

for q in ["What are best practices for post-merger integration?", "How should a bank approach digital transformation?", "What is organizational health and why does it matter?"]:
    result = rag.query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer'][:120]}...")
    print(f"Metrics: {result['metrics']}\n")

---
## Summary

In this session students learned production-grade RAG engineering applied to McKinsey consulting scenarios:

1. **Embeddings** -- How consulting knowledge (strategy reports, engagement briefs, industry analyses) is converted to vectors where semantic similarity equals spatial proximity.
2. **Vector Stores** -- How ChromaDB indexes McKinsey knowledge bases for sub-millisecond similarity search across practices and industries.
3. **Chunking Strategies** -- How splitting strategy (section-aware for reports, sentence-level for summaries) directly affects retrieval quality on consulting documents.
4. **Query Transformation** -- How multi-query and HyDE improve retrieval for complex consulting research questions spanning multiple industries and functions.
5. **End-to-End RAG** -- How to combine retrieval, generation, and source citations into a production pipeline for McKinsey's knowledge management.

**Up Next:** In Session 2, we will learn how to deploy and scale these systems in production.